# M03.02 — When chunking loses the citation

A claims decision must cite the article it applied, for example `CAGA04-E2 Art.18.1.f`. Fixed-size windows know nothing about articles. We measure what that costs, then fix it.

## Setup

The reusable pipeline (`embed`, `chunk_text`, `build_store`, `retrieve`, `answer`, `show`) and splitters live in `agentic_workshop/rag.py`. The cells below rebuild the splitters so you can inspect how article structure affects retrieval.

In [ ]:
import re

from agentic_workshop.rag import answer, build_store, chunk_text, embed, load_documents, retrieve, show

from agentic_workshop.vector_store import MiniVectorStore

documents = load_documents()

# Added to every prompt:
CITE = "Cite the document code and article number for every rule you rely on."

## 0. Ground Truth

Setting a ground truth citation for each invoice. More rigorous ways to evaluate your solution is in a later module. Here, we aim for the article heading and a phrase from the clause that decides the claim, sourced from `data/invoices/ground_truth/`.

In [ ]:
# invoice: (insurance product, treatment on the invoice)
INVOICES = {
    "INV-2025-001": ("Dentaire Plus", "Composite filling, tooth 26"),
    "INV-2025-002": ("Dentaire Plus", "Ceramic crown and bridge, teeth 14-16"),
    "INV-2025-003": ("Dentaire Plus", "Cosmetic teeth whitening"),
    "INV-2025-004": ("Optimum", "Osteopathy session, lower back pain"),
    "INV-2025-005": ("Optimum", "Therapeutic massage, 60 min"),
    "INV-2025-006": ("Optimum", "Thermal cure, 12 days, prescribed by Dr. F. Moret (GP)"),
    "INV-2025-007": ("Optimum", "Non-reimbursable prescribed medication (LPPA list)"),
    "INV-2025-008": ("Optimum", "Rhinoplasty (aesthetic nose correction)"),
    "INV-2025-009": ("Optimum", "Root canal treatment, tooth 36"),
    "INV-2025-010": ("Dentaire Plus", "Composite filling, tooth 14"),
    "INV-2025-011": ("PrimaCare", "Consultation for persistent cough"),
}

# invoice: (document, heading of the cited article, phrase from the deciding clause)
CITATIONS = {
    "INV-2025-001": ("DPAM01-E4", "Art. 3 Benefits", "Class 1: 75%"),
    "INV-2025-002": ("DPAM01-E4", "Art. 3 Benefits", "Class 3: 75%"),
    "INV-2025-003": ("CAGA04-E2", "Art. 18 Exclusions", "cosmetic imperfections"),
    "INV-2025-004": ("SOGA01-E1", "Art. 5 Insured benefits", "75%, max. CHF 3,000"),
    "INV-2025-005": ("SOGA01-E1", "Art. 5 Insured benefits", "for massages"),
    "INV-2025-006": ("SOGA01-E1", "Art. 5 Insured benefits", "90%, max. 30 days"),
    "INV-2025-007": ("SOGA01-E1", "Art. 5 Insured benefits", "Non-reimbursable drugs"),
    "INV-2025-008": ("CAGA04-E2", "Art. 18 Exclusions", "cosmetic imperfections"),
    "INV-2025-009": ("CAGA04-E2", "Art. 18 Exclusions", "for dental treatment, insofar"),
    "INV-2025-010": ("DPAM01-E4", "Art. 4 Entitlement to benefits", "one-month"),
    "INV-2025-011": ("RFBA02-E5", "Art. 7 Non-compliance", "After two reminders"),
}

for invoice, (doc, heading, phrase) in CITATIONS.items():
    text = documents[doc]
    gap = text.index(phrase, text.index(heading)) - text.index(heading)
    print(f"{invoice}  {doc}  {heading:32s} clause starts {gap:5d} characters below the heading")

Some clauses sit about a thousand characters below their heading!

## 1. Naive Chunking

In [ ]:
# Let's naively chunk with chunksize 400, 100 overlap. 
# Are we able to hit the right article? Does it contain the information we need?

chunks = [(name, c) for name, text in documents.items() for c in chunk_text(text, 400, 100)]
print(f"{len(chunks)} chunks\n")

doc, heading, phrase = CITATIONS["INV-2025-001"]
for name, c in chunks:
    if name == doc and phrase in c:
        print(f"--- contains the clause, heading present: {heading in c} ---")
        print(repr(c), "\n")

The clause is there, but the article number and the lead-in ("Coverage is excluded:") are not always there. This chunk may lack the context necessary to be correctly retrieved!

## 2. Retrieval and generation

`INV-2025-006`: a thermal cure under Optimum.

In [ ]:
# Let's try to ask a question that should be answerable. Do we get the information we ask for?

def claim_question(invoice: str) -> str:
    product, treatment = INVOICES[invoice]
    return f"Is '{treatment}' covered under {product}?"


store = build_store(documents, lambda text: chunk_text(text, 400, 100))
question = claim_question("INV-2025-005")

print(question)
show(retrieve(store, question, k=3))

In [ ]:
print(answer(store, question, k=3, instruction=CITE))

Is the citation actually there? Sometimes a paragraph number gets cited as if it were the article. If nothing is cited, why not?

## 3. Can we tune our way out?

Do some chunks contain both the heading and the clause? Maybe we just need to find the right settings.

In [ ]:
def citable(pieces: list[tuple[str, str]]) -> int:
    """How many invoices have their article heading and their clause in the same chunk?"""

    return sum(
        any(name == doc and heading in c and phrase in c for name, c in pieces)
        for doc, heading, phrase in CITATIONS.values()
    )


print(f"{'size':>6} {'overlap':>8} {'chunks':>7} {'citable':>9}")
for size in [400, 600, 800, 1000, 1200, 1600, 2000, 3000]:
    for ov in [0, 100, 200]:
        cs = [(n, c) for n, t in documents.items() for c in chunk_text(t, size, ov)]
        print(f"{size:6d} {ov:8d} {len(cs):7d} {citable(cs):7d}/{len(CITATIONS)}")

- The score jumps around with chunk size. Good scores are pure luck, and there's not guaranteed to be a chunksize + overlap setting that works everywhere.
- Overlap does not help a clause that sits far below its heading.

## 5. Splitting on structure

Split at article headings, matching only at the start of a line. We did this at the end of a previous notebook. Now we have measure how and why it helps.

In [ ]:
ARTICLE_HEADING = re.compile(r"(?m)^Art\. \d+[a-z]? +[A-Z][^\n]*")


def split_on_articles(text: str) -> list[str]:
    """Split at article headings, keeping each heading with its article."""
    starts = [m.start() for m in ARTICLE_HEADING.finditer(text)]
    return [text[a:b].strip() for a, b in zip(starts, starts[1:] + [len(text)], strict=True)]


articles = [(n, a) for n, t in documents.items() for a in split_on_articles(t)]
sizes = sorted(len(a) for _, a in articles)

print(f"{len(articles)} articles, {sizes[0]} to {sizes[-1]} characters "
      f"(median {sizes[len(sizes) // 2]})")
print(f"citable: {citable(articles)}/{len(CITATIONS)}")

Every invoice is citable! Now let's retrieve.

In [ ]:
article_store = build_store(documents, split_on_articles)

show(retrieve(article_store, question, k=3))

In [ ]:
print(answer(article_store, question, k=3, instruction=CITE))

Is it better sourced now? Try with different questions, do we see improvement? Are there still (a lot of) mistakes?

## 5. Structure *and* size

Splitting on articles creates some very large and some very small chunks. The large ones may match more easily, since they may contain more context, even though for some questions the smaller chunks actually contain the answer. We now split on articles, window the long ones, and repeat the heading to preserve the context.

In [ ]:
def split_structured(text: str, max_chars: int = 800, overlap: int = 100) -> list[str]:
    """Split on articles, then window long articles, repeating the heading on every piece."""

    pieces = []
    for article in split_on_articles(text):
        heading, _, body = article.partition("\n")
        if len(article) <= max_chars:
            pieces.append(article)
            continue
        for window in chunk_text(body, max_chars - len(heading) - 1, overlap):
            pieces.append(f"{heading}\n{window}")
    return pieces


structured = [(n, p) for n, t in documents.items() for p in split_structured(t)]
print(f"{len(structured)} pieces, citable: {citable(structured)}/{len(CITATIONS)}")

structured_store = build_store(documents, split_structured)

show(retrieve(structured_store, question, k=3))

In [ ]:
print(answer(structured_store, question, k=3, instruction=CITE))

Compare the three strategies on the different questions. Is this improving things?

In [ ]:
def retrievable(store: MiniVectorStore, k: int = 3) -> int:
    """How many invoices get a chunk with both heading and clause in the top k?"""

    questions = [claim_question(inv) for inv in CITATIONS]
    hits = 0
    for (doc, heading, phrase), vector in zip(CITATIONS.values(), embed(questions), strict=True):
        hits += any(
            r["metadata"]["source"] == doc and heading in r["text"] and phrase in r["text"]
            for r in store.query(vector, k=k)
        )
    return hits


n = len(CITATIONS)
print(f"{'strategy':22s} {'chunks':>7} {'citable':>8} {'retrievable':>12} {'chars/chunk':>12}")
for label, s in [("fixed 400 / 100", store), ("articles", article_store),
                 ("articles, max 800", structured_store)]:
    pieces = list(zip([m["source"] for m in s.metadata], s.texts, strict=True))
    print(f"{label:22s} {len(s):7d} {citable(pieces):6d}/{n} {retrievable(s):10d}/{n} "
          f"{sum(map(len, s.texts)) // len(s):12d}")

- The structured split wins on both citability and retrievability.
- The remaining misses are exclusions: "is X covered?" against "coverage is excluded for…". This is not a chunking problem (see M04)!
- Bigger pieces cost more tokens per hit, but it remains tiny compared to stuffing the entire corpus in the context.

## Takeaways

- Chunking decides what can be retrieved and cited.
- Fixed windows can keep the clause but drop the article number. The model then says it cannot cite, or cites a wrong number that looks right.
- No chunk size works in general; tuning only moves the failure around.
- Splitting on articles makes everything citable, but long articles dilute and retrieval can get worse.
- Better: use the document's structure, parse it carefully, cap the size and repeat the heading.
- Check chunks directly: `citable` just requires some knowledge of the source documents.

## Extensions

**Easy**

1. **Widen the sweep.** Plot `citable` and `retrievable` over size × overlap, and add context size as a cost. Also sweep `max_chars`.
2. **Verifiable citations.** Require a verbatim quote plus the article number, then write `verify_citation` to check that the quote really lies inside that article. Tip: use pydantic or similar for structured output!

**Medium**

3. **Hierarchical splits.** Split on paragraphs and list items, i.e. `CAGA04-E2 › Art. 18 › 1 › f`. At each level, keep the lead-in. This is similar to how we repeated the header before, but now at each hierarchical split. Similarly split Optimum's table into rows. For cleanup, also completely strip page footers. Notice how the splits become more fine-grained, with many repeated headers. This works for some documents, but may decrease performance for others. Does it help in this case?
4. **Parent-document retrieval.** We can revisit fixed size chunking, alleviating the context-answer break by repeating the parent article for each chunk. Does this improve retrievability? What are the drawbacks?

**Hard**

5. **Contextual retrieval.** Have the LLM write one sentence of context per chunk, based on the surrounding pages, before embedding it. Measure the gain and the indexing cost, and compare with the structured splitter.

In [ ]:
# Extension 1: widen the sweep
from functools import partial

import matplotlib.pyplot as plt


def scores(splitter, k: int = 3) -> tuple[int, int, float]:
    """return citable, retrievable, and approximate token cost."""

    temp_store = build_store(documents, splitter)

    sources = [meta["source"] for meta in temp_store.metadata]
    citable_count = citable(list(zip(sources, temp_store.texts)))
    retrievable_count = retrievable(temp_store, k)
    token_cost = k * sum(map(len, temp_store.texts)) / len(temp_store) / 4

    return citable_count, retrievable_count, token_cost


sizes = [400, 600, 800, 1200, 2000, 3000]
curves = {}
for overlap in [0, 100, 200]:
    chunker_scores = []
    for size in sizes:
        chunker_to_score = partial(chunk_text, chunk_size=size, overlap=overlap)
        chunker_scores.append(scores(chunker_to_score))
    curves[f"fixed, overlap {overlap}"] = chunker_scores
curves["structured, max_chars"] = [scores(partial(split_structured, max_chars=size)) for size in sizes]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for label, results in curves.items():
    per_metric = zip(*results)  # [(citable, retrievable, tokens), ...] -> (citables, retrievables, tokens)
    for ax, values in zip(axes, per_metric):
        ax.plot(sizes, values, marker="o", label=label)
for ax, title in zip(axes, ["citable", "retrievable (top 3)", "prompt tokens (top 3)"]):
    ax.set(title=title, xlabel="chunk size or max_chars (characters)")
axes[0].legend();

In [ ]:
# Extension 2: verifiable citations
from pydantic import BaseModel, Field

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL

# Here, we use pydantic for structured output!
class Citation(BaseModel):
    document: str = Field(description="Document code, e.g. CAGA04-E2")
    article: str = Field(description="Article number only, e.g. 18 or 3a")
    quote: str = Field(description="The rule, copied verbatim from the extracts")


class CitedAnswer(BaseModel):
    answer: str
    citations: list[Citation]


def verify_citation(c: Citation) -> bool:
    """Is the quote inside that article? Compared on letters and digits only, so PDF line breaks and hyphens don't matter."""
    squash = lambda s: re.sub(r"\W+", "", s.lower())
    return any(a.startswith(f"Art. {c.article} ") and squash(c.quote) in squash(a)
               for a in split_on_articles(documents.get(c.document, "")))


for invoice in CITATIONS:
    q = claim_question(invoice)
    context = "\n\n".join(f"[{r['metadata']['source']}]\n{r['text']}" for r in retrieve(structured_store, q, k=3))
    prompt = f"Answer the question using only the extracts below. {CITE} Quote each rule verbatim.\n\nEXTRACTS\n{context}\n\nQUESTION\n{q}"
    cited = get_client().responses.parse(model=CHAT_MODEL, input=prompt, text_format=CitedAnswer).output_parsed
    print(f"{invoice}  {q}  (expected: {CITATIONS[invoice][0]} {CITATIONS[invoice][1]})")
    for c in cited.citations:
        print(f"    {'ok  ' if verify_citation(c) else 'FAIL'}  {c.document} Art. {c.article}: {c.quote[:70]}")
# A verified quote is real, but not necessarily the most relevant one. 
# It allows the reader to quickly inspect whether a summary is well supported by the citations, 
# but not whether the retrieval was exhaustive or, if it failed, whether the text doesn't exist or the similarity search simply failed.

In [ ]:
# Extension 3: below the article
FOOTER = re.compile(r"(?m)^(Compan(y|ies) under Groupe Mutuel|Administrative details:|\w+-E\d – Edition:|Coverage Description).*\n?")
PARAGRAPH, ITEM = re.compile(r"(?m)^ *(\d+)\. "), re.compile(r"(?m)^ *([a-z])\. ")
TABLE_ROW = re.compile(r"(?<=\.\n)(?=[A-Z])")  # Optimum's table: a new row starts after a line ending in "."


def split_at(pattern: re.Pattern, text: str) -> tuple[str, list[tuple[str, str]]]:
    """The text before the first match (the lead-in), and (label, piece) for every match."""
    ms = list(pattern.finditer(text))
    ends = [m.start() for m in ms[1:]] + [len(text)]
    return text[:ms[0].start()] if ms else text, [(m.group(1), text[m.start():e]) for m, e in zip(ms, ends)]


def split_fine(doc: str, text: str) -> list[str]:
    """One piece per list item, paragraph or table row, each with its breadcrumb, heading and lead-in."""
    pieces = []
    for article in split_on_articles(FOOTER.sub("", text)):
        heading, _, body = article.partition("\n")
        art = " ".join(heading.split()[:2])
        if (doc, art) == ("SOGA01-E1", "Art. 5"):
            rows = [(row.split("\n")[0].strip(), row) for row in TABLE_ROW.split(body)]
        else:
            rows = split_at(PARAGRAPH, body)[1] or [("", body)]
        for label, para in rows:
            lead, items = split_at(ITEM, para)
            for item, piece in items or [("", "")]:
                crumb = " › ".join(x for x in [doc, art, label, item] if x)
                pieces.append(f"{crumb}\n{heading}\n{lead if (doc, art) != ('SOGA01-E1', 'Art. 5') else para}{piece}".strip())
    return pieces


fine = [(n, p) for n, t in documents.items() for p in split_fine(n, t)]
fine_store = MiniVectorStore()
fine_store.add(embed([p for _, p in fine]), [p for _, p in fine], [{"source": n} for n, _ in fine])
print(f"{len(fine)} pieces, citable {citable(fine)}/{len(CITATIONS)}, retrievable {retrievable(fine_store)}/{len(CITATIONS)}\n")
print(next(p for _, p in fine if "cosmetic imperfections" in p))

In [ ]:
# Extension 4: parent-document retrieval
windows = [(n, a, w) for n, t in documents.items() for a in split_on_articles(t) for w in chunk_text(a, 300, 50)]
window_store = MiniVectorStore()
window_store.add(embed([w for *_, w in windows]), [w for *_, w in windows], [{"source": n, "parent": a} for n, a, _ in windows])


def retrieve_parents(question: str, k: int = 3) -> list[tuple[str, str]]:
    """The parent articles of the best-matching windows, the first k distinct ones."""
    hits = window_store.query(embed([question])[0], k=30)
    return list(dict.fromkeys((h["metadata"]["source"], h["metadata"]["parent"]) for h in hits))[:k]


results = {inv: retrieve_parents(claim_question(inv)) for inv in CITATIONS}
hits = sum(any(n == doc and heading in a and phrase in a for n, a in results[inv]) for inv, (doc, heading, phrase) in CITATIONS.items())
tokens = sum(len(a) for parents in results.values() for _, a in parents) / len(results) / 4
print(f"parent-document: retrievable {hits}/{len(CITATIONS)}, ~{tokens:.0f} prompt tokens per question")
print(f"structured:      retrievable {retrievable(structured_store)}/{len(CITATIONS)}, "
      f"~{3 * sum(map(len, structured_store.texts)) / len(structured_store) / 4:.0f} prompt tokens per question")
# It breaks on long parents: one hit in Optimum's table (Art. 5, ~8'500 characters) puts the whole table in the prompt.
# It's also quite a bit more expensive.

In [ ]:
# Extension 5: contextual retrieval
from concurrent.futures import ThreadPoolExecutor

from agentic_workshop.config import REASONING_OFF
from agentic_workshop.costs import print_cost


def situate(doc: str, chunk: str):
    # The document comes first, so calls share a cacheable prefix (see `cached` below; parallel first calls may miss it).
    prompt = f"""<document>\n{documents[doc]}\n</document>\n<chunk>\n{chunk}\n</chunk>
In one sentence, situate the chunk within the document to improve search retrieval of the chunk.
Start with the document code and the heading of the article the chunk belongs to, e.g. "{doc} Art. 3 Benefits: ...".
Output only the sentence."""
    return get_client().responses.create(model=CHAT_MODEL, input=prompt, reasoning=REASONING_OFF)


fixed = [(n, c) for n, t in documents.items() for c in chunk_text(t, 400, 100)]
with ThreadPoolExecutor(8) as pool:
    responses = list(pool.map(lambda p: situate(*p), fixed))

texts = [f"{r.output_text}\n{c}" for r, (_, c) in zip(responses, fixed, strict=True)]
contextual_store = MiniVectorStore()
contextual_store.add(embed(texts), texts, [{"source": n} for n, _ in fixed])

for label, s in [("fixed 400 / 100", store), ("fixed + context", contextual_store), ("structured", structured_store)]:
    print(f"{label:16s} retrievable {retrievable(s)}/{len(CITATIONS)}")
print("\nIndexing cost of the context sentences:")
print_cost([r.usage for r in responses])
# The heading now comes from the model rather than repeated from the text!
# The model can be wrong, so this creates a new need for observability.